# Free Search Audit — Prospect Template

**Use this with:** any Indian mid-market auto-parts platform considering our search API.

**Inputs needed from prospect:**
1. `catalog.csv` — their product catalog. Required columns: `id`, `name`. Useful extras: `category`, `brand`, `vehicle`, `description`, `part_number`.
2. `queries.csv` — their top N search queries from the last 30 days. Required columns: `query`, `count`. Useful extras: `zero_result_flag`, `conversion_rate`.

**Output delivered to prospect:**
- PDF report with zero-result queries their search failed on that semantic search catches, estimated revenue leak, per-query before/after comparison, ~20 worst zero-result queries.

**Runtime:** ~30 min for 10K catalog + 500 queries on `text-embedding-3-large`.

See ADR 011 (context/decisions/011-positioning-moat.md) for why this is the GTM wedge.

## 0. Setup

In [ ]:
# pip install openai pandas numpy scikit-learn tqdm tiktoken
import os
import pandas as pd
import numpy as np
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
MODEL = 'text-embedding-3-large'  # 3072 dims, strong on multilingual + semantic
PROSPECT_NAME = 'ACME_Auto_Parts'
OUTPUT_DIR = f'./audit_output/{PROSPECT_NAME}/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Load prospect data

In [ ]:
catalog = pd.read_csv(f'{OUTPUT_DIR}/catalog.csv')
queries = pd.read_csv(f'{OUTPUT_DIR}/queries.csv')

# Normalize: search_text concatenates the useful fields
def product_text(row):
    parts = [str(row.get('name', ''))]
    for col in ['category', 'brand', 'vehicle', 'part_number', 'description']:
        v = row.get(col)
        if pd.notna(v) and v != '':
            parts.append(str(v))
    return ' | '.join(parts)

catalog['search_text'] = catalog.apply(product_text, axis=1)
print(f'Catalog: {len(catalog)} products')
print(f'Queries: {len(queries)} ({queries.get("zero_result_flag", pd.Series()).sum()} zero-result)')
catalog.head(3)

## 2. Embed catalog (batched, cached)

In [ ]:
def embed_batch(texts, batch_size=100):
    out = []
    for i in tqdm(range(0, len(texts), batch_size), desc='embedding'):
        batch = texts[i:i+batch_size]
        resp = client.embeddings.create(model=MODEL, input=batch)
        out.extend([d.embedding for d in resp.data])
    return np.array(out)

cache_path = f'{OUTPUT_DIR}/catalog_embeddings.npy'
if os.path.exists(cache_path):
    catalog_emb = np.load(cache_path)
    print(f'loaded cached embeddings: {catalog_emb.shape}')
else:
    catalog_emb = embed_batch(catalog['search_text'].tolist())
    np.save(cache_path, catalog_emb)
    print(f'cached to {cache_path}')

## 3. For each query: semantic top-10 + detect gaps vs prospect's current search

**Assumption:** if `queries.csv` has `zero_result_flag=True`, their current search returned nothing. Our semantic search returns the top-10 by cosine similarity. These are the queries where we can most visibly outperform.

**Stretch (if time permits):** run their actual catalog through a simple BM25 tokenizer on `search_text` for side-by-side ranking comparison. Not required for v0 audit.

In [ ]:
query_texts = queries['query'].astype(str).tolist()
query_emb = embed_batch(query_texts)

# Cosine similarity: (num_queries, num_products)
sims = cosine_similarity(query_emb, catalog_emb)
TOP_K = 10

results = []
for qi, q in enumerate(query_texts):
    top_idx = sims[qi].argsort()[::-1][:TOP_K]
    top_products = catalog.iloc[top_idx].copy()
    top_products['score'] = sims[qi][top_idx]
    top_products['rank'] = range(1, TOP_K + 1)
    top_products['query'] = q
    results.append(top_products[['query', 'rank', 'id', 'name', 'score']])

all_results = pd.concat(results, ignore_index=True)
all_results.to_csv(f'{OUTPUT_DIR}/semantic_top10.csv', index=False)
print(f'wrote {OUTPUT_DIR}/semantic_top10.csv ({len(all_results)} rows)')

## 4. Zero-result rescue analysis — the headline for the audit report

In [ ]:
# Which zero-result queries have plausible semantic matches (top-1 score > threshold)?
THRESHOLD = 0.5  # tune per prospect; 0.5 is usually meaningful semantic similarity

zero_results = queries[queries.get('zero_result_flag', False) == True].copy()
zero_results = zero_results.merge(
    all_results[all_results['rank'] == 1].rename(columns={'score': 'top1_score', 'name': 'top1_match'}),
    left_on='query', right_on='query', how='left'
)

rescuable = zero_results[zero_results['top1_score'] >= THRESHOLD]
rescue_rate = len(rescuable) / max(len(zero_results), 1)

print(f'Zero-result queries: {len(zero_results)}')
print(f'Rescued by semantic search (score >= {THRESHOLD}): {len(rescuable)} ({rescue_rate:.1%})')
print()
print('Top 10 rescued queries by search volume:')
rescuable.sort_values('count', ascending=False).head(10)[['query', 'count', 'top1_match', 'top1_score']]

## 5. Revenue leak estimate (prospect-specific assumptions)

In [ ]:
# Assumptions — override per prospect
AVG_ORDER_VALUE_INR = 1500   # prospect tells us; industry default for auto parts
BASELINE_CONVERSION = 0.03   # 3% — conservative for failed-search users
ZERO_RESULT_CONVERSION = 0   # zero-result queries convert at 0 by definition
RECOVERY_FRACTION = 0.5      # fraction of rescued-query users we'd actually convert

rescued_searches = rescuable['count'].sum()
recovered_users = rescued_searches * RECOVERY_FRACTION
monthly_revenue_leak = recovered_users * BASELINE_CONVERSION * AVG_ORDER_VALUE_INR
annual_revenue_leak = monthly_revenue_leak * 12

print(f'Monthly revenue leak (estimated): Rs.{monthly_revenue_leak:,.0f}')
print(f'Annual revenue leak (estimated):  Rs.{annual_revenue_leak:,.0f}')
print(f'Our proposed annual fee:          Rs.{25_000 * 12:,}')
print(f'ROI vs fee:                        {annual_revenue_leak / (25_000 * 12):.1f}x')

## 6. Export the audit report (markdown → PDF via pandoc)

In [ ]:
report_md = f'''# Search Audit — {PROSPECT_NAME}

Generated: {pd.Timestamp.now().strftime('%Y-%m-%d')}

## Summary

| Metric | Value |
|--------|-------|
| Catalog size | {len(catalog):,} products |
| Queries analyzed | {len(queries):,} (30 days) |
| Zero-result queries | {len(zero_results):,} |
| **Rescued by semantic search** | **{len(rescuable):,} ({rescue_rate:.1%})** |
| **Estimated monthly revenue leak** | **Rs.{monthly_revenue_leak:,.0f}** |
| **Estimated annual revenue leak** | **Rs.{annual_revenue_leak:,.0f}** |

## Top 20 queries we can fix that you currently return zero results for

{rescuable.sort_values('count', ascending=False).head(20)[['query', 'count', 'top1_match', 'top1_score']].to_markdown(index=False)}

## Methodology

- Embeddings via OpenAI `text-embedding-3-large`.
- For each query, top-10 products by cosine similarity.
- A query is "rescuable" if your search returned zero results AND our top-1 semantic match has score >= {THRESHOLD}.
- Revenue leak = rescued query volume × 50% recovery × 3% conversion × Rs.{AVG_ORDER_VALUE_INR} AOV.

## Proposed next step

1. Pilot: we deploy a search endpoint against your full catalog for 30 days at no cost.
2. A/B test: 10% of traffic → our endpoint, 90% → your current search.
3. Decide based on measured conversion lift.

Target pricing after pilot: Rs.25K/month at your scale — ~{annual_revenue_leak / (25_000 * 12):.0f}x estimated ROI.
'''

report_path = f'{OUTPUT_DIR}/audit_report.md'
open(report_path, 'w').write(report_md)
print(f'wrote {report_path}')
# To render to PDF: pandoc audit_report.md -o audit_report.pdf --pdf-engine=weasyprint

## 7. What's NOT in this audit (for honest framing with prospect)

- No Hindi/Hinglish detection — our fine-tuned model would handle this; this audit uses general-purpose OpenAI embeddings which are good-not-great on Hinglish.
- No fitment filter — our KG has 688 `fits` edges (NHTSA-derived); a pilot would enrich from their catalog.
- No reranker / query classifier — these are Phase 5 items; the audit shows the floor, not the ceiling.
- Revenue leak estimate uses industry assumptions — over-rides welcome.

**Use this gap as the sales handoff:** "these are the results with off-the-shelf embeddings. Our domain-specific model + KG fitment + Hindi tokenizer ships X% better. Want to pilot?"